# schedule dashboard page refresh v1

目标：验证 `src/schedule` 可以把模型矩阵刷新、dashboard export、view model 和 HTML dashboard 页面串成一条可定时执行的链路。

In [1]:
from pathlib import Path
import json
import os
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'SPEC.md').exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError('cannot locate repo root from notebook cwd')
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.schedule import ModelDashboardPipelineConfig, run_model_dashboard_refresh_once

CARD = Path('research/schedule_dashboard_page_refresh_v1')
MATRIX_CARD = CARD / 'matrix_card'
DASHBOARD_EXPORT = CARD / 'dashboard_export.json'
DASHBOARD_HTML = CARD / 'dashboard.html'
ATTRIBUTION_CARD = Path('research/model_attribution_analysis_v1')
LOCK_DIR = CARD / 'locks'
CARD.mkdir(parents=True, exist_ok=True)

{'card': str(CARD), 'dashboard_html': str(DASHBOARD_HTML)}

{'card': 'research/schedule_dashboard_page_refresh_v1',
 'dashboard_html': 'research/schedule_dashboard_page_refresh_v1/dashboard.html'}

## 执行 schedule refresh

In [2]:
config = ModelDashboardPipelineConfig(
    name='schedule_dashboard_page_refresh_v1',
    card_dir=MATRIX_CARD,
    dashboard_output_path=DASHBOARD_EXPORT,
    dashboard_page_output_path=DASHBOARD_HTML,
    dashboard_page_title='SOXL 模型回测 Dashboard',
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse', 'gcn_mse'),
    top_n_factors=3,
    dashboard_top_n=5,
    attribution_card_dir=ATTRIBUTION_CARD,
    retries=1,
)
events = []
result = run_model_dashboard_refresh_once(config, lock_dir=LOCK_DIR, notify=events.append)
{'status': result.status, 'payload': result.payload, 'events': events}

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'success',
 'payload': {'matrix_card': 'research/schedule_dashboard_page_refresh_v1/matrix_card',
  'dashboard_export': 'research/schedule_dashboard_page_refresh_v1/dashboard_export.json',
  'dashboard_schema_version': 'dashboard.model_matrix.v1',
  'dashboard_page': 'research/schedule_dashboard_page_refresh_v1/dashboard.html',
  'dashboard_page_schema_version': 'dashboard.static_page.v1',
  'dashboard_view_model_schema_version': 'dashboard.view_model.v1',
  'dashboard_page_model_card_count': 5,
  'top_model': 'ridge_mse__top_ic_factors',
  'attribution_card': 'research/model_attribution_analysis_v1',
  'attribution_status': 'completed',
  'matrix_status': 'feature_set_matrix_completed',
  'variant_count': 6,
  'attempts': 1},
 'events': ['schedule_dashboard_page_refresh_v1:success']}

## 验证页面链路

In [3]:
dashboard_payload = json.loads(DASHBOARD_EXPORT.read_text(encoding='utf-8'))
html = DASHBOARD_HTML.read_text(encoding='utf-8')
summary = {
    'schema_version': 'schedule.dashboard_page_refresh.v1',
    'status': result.status,
    'job_name': config.name,
    'attempts': result.payload.get('attempts'),
    'notifications': events,
    'matrix_card': str(MATRIX_CARD),
    'dashboard_export': str(DASHBOARD_EXPORT),
    'dashboard_page': str(DASHBOARD_HTML),
    'dashboard_schema_version': result.payload.get('dashboard_schema_version'),
    'dashboard_page_schema_version': result.payload.get('dashboard_page_schema_version'),
    'dashboard_view_model_schema_version': result.payload.get('dashboard_view_model_schema_version'),
    'dashboard_page_model_card_count': result.payload.get('dashboard_page_model_card_count'),
    'html_size_bytes': DASHBOARD_HTML.stat().st_size,
    'top_models': [row['model'] for row in dashboard_payload['top_models']],
    'attribution_status': dashboard_payload['attribution']['status'],
    'has_model_filter': 'id="model-filter"' in html,
    'has_regime_filter': 'id="regime-filter"' in html,
    'has_fidelity_filter': 'id="fidelity-filter"' in html,
    'has_view_model_json': 'id="dashboard-view-model"' in html,
    'has_window_regime_table': 'window-regime' in html,
    'risk_notes': [
        '页面由 schedule 自动刷新生成，仍只消费 dashboard export/view model。',
        '当前是 paper/sim dashboard 页面，不包含真实 broker 下单能力。',
    ],
}
assert summary['status'] == 'success'
assert summary['dashboard_page_schema_version'] == 'dashboard.static_page.v1'
assert summary['dashboard_view_model_schema_version'] == 'dashboard.view_model.v1'
assert summary['dashboard_page_model_card_count'] == len(dashboard_payload['top_models'])
assert summary['has_model_filter']
assert summary['has_regime_filter']
assert summary['has_fidelity_filter']
assert summary['has_view_model_json']
assert summary['has_window_regime_table']
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'schedule.dashboard_page_refresh.v1',
 'status': 'success',
 'job_name': 'schedule_dashboard_page_refresh_v1',
 'attempts': 1,
 'notifications': ['schedule_dashboard_page_refresh_v1:success'],
 'matrix_card': 'research/schedule_dashboard_page_refresh_v1/matrix_card',
 'dashboard_export': 'research/schedule_dashboard_page_refresh_v1/dashboard_export.json',
 'dashboard_page': 'research/schedule_dashboard_page_refresh_v1/dashboard.html',
 'dashboard_schema_version': 'dashboard.model_matrix.v1',
 'dashboard_page_schema_version': 'dashboard.static_page.v1',
 'dashboard_view_model_schema_version': 'dashboard.view_model.v1',
 'dashboard_page_model_card_count': 5,
 'html_size_bytes': 126723,
 'top_models': ['ridge_mse__top_ic_factors',
  'gcn_mse__top_ic_factors',
  'gcn_mse__ic_weighted_composite',
  'ridge_mse__ic_weighted_composite',
  'ridge_mse__all_features'],
 'attribution_status': 'completed',
 'has_model_filter': True,
 'has_regime_filter': True,
 'has_fidelity_filt

## 结论

- 做了什么：通过 `src/schedule` 默认流程生成 matrix card、dashboard export 和 `dashboard.html`。
- 核心指标：job status、attempts、dashboard/page schema、模型卡片数量、HTML size 以 `analysis_summary.json` 为准。
- regime 差异：页面包含 window-regime 表和 model_effect_deltas，可展示分 regime 和分窗口差异。
- 是否晋升：这是自动刷新链路验证，不是模型晋升。
- 风险：页面为 paper/sim dashboard；`gcn_mse` 仍是 proxy_feature_adapter，不能作为论文级图模型实盘结论。
- 下一步：补 production notebook，把 paper intent 和 dashboard page refresh 固化为可运行的 paper/sim 生产候选流程。